# Create Embeddings with CNN embed_150

In this notebook, we use dataset created by @ummaymariamuna and embed all the videos from that dataset using custom CNN model.

Dataset Structure:

```
/train
|- anomalous
|- nonanomalous
```



In [1]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

In [2]:
# Set the device to use (e.g., 'cpu', 'cuda', 'mps')
device = torch.device("mps" if torch.backends.mps.is_available() else
                      ("cuda" if torch.cuda.is_available() else "cpu"))

# Select Device According to Availability
print("Device selected:", device)

# If the device is CUDA, print the device capability
if device.type == "cuda":
    os.system("nvidia-smi")
    print()
    print("Device type:", device.type)
    print("Capability:", torch.cuda.get_device_capability(device))
else:
    print("Device capabilities are limited on MPSs and CPUs.")

Device selected: cuda
Thu Jul  4 19:03:32 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             26W /  250W |       3MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------

In [3]:
# import torch
# import torch.nn as nn
# import timm
# import cv2
# import numpy as np
# from tqdm import tqdm

# # Define the device
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# class VideoFeatureExtractor(nn.Module):
#     def __init__(self, vit_model='vit_small_patch16_224', output_dim=2048):  # Adjusted to 2048
#         super(VideoFeatureExtractor, self).__init__()
#         self.vit = timm.create_model(vit_model, pretrained=True)
#         self.vit.reset_classifier(0)
        
#         vit_feature_dim = self.vit.num_features
#         self.spatial_projection = nn.Linear(vit_feature_dim, output_dim // 2)
#         self.flow_projection = nn.Linear(2 * 224 * 224, output_dim // 2)  # Assuming 224x224 input
        
#     def forward(self, frame, optical_flow):
#         # Spatial features
#         vit_features = self.vit.forward_features(frame)
#         spatial_features = self.spatial_projection(vit_features[:, 0])
        
#         # Motion features
#         flow_flattened = optical_flow.reshape(optical_flow.size(0), -1)
#         flow_features = self.flow_projection(flow_flattened)
        
#         # Combine spatial and motion features
#         combined_features = torch.cat([spatial_features, flow_features], dim=1)
#         return combined_features

# # Function to calculate optical flow
# def calculate_optical_flow(prev_frame, curr_frame):
#     prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_RGB2GRAY)
#     curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_RGB2GRAY)
#     flow = cv2.calcOpticalFlowFarneback(prev_gray, curr_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
#     return flow

# # Function to extract features from video frames
# def extract_video_features(video_frames, model, device, output_dim=2048):
#     video_frames = torch.from_numpy(video_frames).permute(0, 3, 1, 2).to(device)
    
#     features = []
#     prev_frame = None
#     for i in range(len(video_frames)):
#         frame = video_frames[i]
        
#         # Convert frame from [C, H, W] to [H, W, C] and then to numpy array
#         frame_np = frame.permute(1, 2, 0).cpu().numpy()
        
#         if prev_frame is not None:
#             optical_flow = calculate_optical_flow(prev_frame, frame_np)
#             optical_flow = torch.from_numpy(optical_flow).permute(2, 0, 1).unsqueeze(0).float().to(device)
            
#             # Ensure the optical flow tensor is contiguous
#             optical_flow = optical_flow.contiguous()
            
#             # Convert frame back to [C, H, W] for the ViT model
#             frame_tensor = frame.unsqueeze(0)
            
#             frame_features = model(frame_tensor, optical_flow)
#             features.append(frame_features.squeeze(0))  # Remove batch dimension
        
#         prev_frame = frame_np
    
#     if features:
#         features = torch.stack(features)
#     else:
#         features = torch.tensor([], device=device)

#     return features.cpu().detach().numpy()

# # Initialize the feature extractor model
# model = VideoFeatureExtractor(output_dim=2048).to(device)
# model.eval()

In [4]:
import torch
import torch.nn as nn
import timm
import cv2
import numpy as np
import glob
from tqdm import tqdm
from random import shuffle

# Define the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class VideoFeatureExtractor(nn.Module):
    def __init__(self, vit_model='vit_small_patch16_224', output_dim=2048):  # Adjusted to 768 for ViT small
        super(VideoFeatureExtractor, self).__init__()
        self.vit = timm.create_model(vit_model, pretrained=True)
        self.vit.reset_classifier(0)
        
        vit_feature_dim = self.vit.num_features
        self.spatial_projection = nn.Linear(vit_feature_dim, output_dim)
        
    def forward(self, frame):
        # Spatial features
        vit_features = self.vit.forward_features(frame)
        spatial_features = self.spatial_projection(vit_features[:, 0])
        
        return spatial_features

    
# Function to extract features from video frames
def extract_video_features(video_frames, model, device, output_dim=2048):
    video_frames = torch.from_numpy(video_frames).permute(0, 3, 1, 2).to(device)
    
    features = []
    for i in range(len(video_frames)):
        frame = video_frames[i].unsqueeze(0)  # Add batch dimension
        frame_features = model(frame)
        features.append(frame_features.squeeze(0))  # Remove batch dimension
    
    if features:
        features = torch.stack(features)
    else:
        features = torch.tensor([], device=device)
        
    return features.cpu().detach().numpy()

# Initialize the feature extractor model
model = VideoFeatureExtractor(output_dim=2048).to(device)
model.eval()


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

VideoFeatureExtractor(
  (vit): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate=

In [5]:
def get_frames(path):
    video = cv2.VideoCapture(path)
    if not video.isOpened():
        raise IOError("Error reading video file")

    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    frames = []
    for i in range(0, frame_count, 2):
        video.set(1, i)
        ret, frame = video.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))
            frame = frame.astype("float32") / 255.0
            frames.append(frame)
    video.release()
    return np.array(frames)

In [6]:
import numpy as np
import glob
from random import shuffle
from tqdm import tqdm  # Ensure tqdm is imported properly

# Your existing code snippet
train_path = '/kaggle/input/rfw-2000-dataset/RWF-2000/*/*.mp4'
train_paths = glob.glob(train_path)
train_labels = [1 if i.split('/')[5] == 'Fight' else 0 for i in train_paths]

train_zip = list(zip(train_paths, train_labels))
shuffle(train_zip)
train_paths, train_labels = zip(*train_zip)

train_labels = list(train_labels)


In [7]:
# Assume train_paths and train_labels are defined
all_video_features = []
all_labels = []

for path, label in tqdm(zip(train_paths, train_labels), total=len(train_paths)):
    try:
        frames = get_frames(path)
        features = extract_video_features(frames, model, device, output_dim=2048)
        all_video_features.append(features)
        all_labels.append(label)
        print(features.shape)
        # Process or save features as needed
    except Exception as e:
        print(f"Error processing {path}: {e}")

# Convert lists to numpy arrays for further processing or saving
all_video_features = np.array(all_video_features)
all_labels = np.array(all_labels)

print("Feature extraction completed.")
np.save('/kaggle/working/rfw_vitof_embeddings.npy', all_video_features) 
np.save('/kaggle/working/rfw_vitof_labels.npy', all_labels)

  0%|          | 1/2000 [00:01<56:32,  1.70s/it]

(75, 2048)


  0%|          | 2/2000 [00:02<43:10,  1.30s/it]

(75, 2048)


  0%|          | 3/2000 [00:04<52:46,  1.59s/it]

...

100%|█████████▉| 1999/2000 [45:29<00:01,  1.47s/it]

(75, 2048)


100%|██████████| 2000/2000 [45:33<00:00,  1.37s/it]

(75, 2048)


Feature extraction completed.


In [8]:
print("Shape of all_video_features:", all_video_features.shape)
print("Shape of all_labels:", all_labels.shape)

Shape of all_video_features: (2000, 75, 2048)
Shape of all_labels: (2000,)
